# 📈 Quantitative Backtesting Framework: Trend-Following vs. Buy & Hold

## Overview
Professional-grade OOP backtesting engine for evaluating algorithmic trading strategies across decades of historical data.

## Core Capabilities
* **Parallel Rolling Windows:** Hundreds of sequential simulations run in parallel via `ThreadPoolExecutor`.
* **Next-Day Open Execution:** Entry at next-day open (open→close return). Exit at open (overnight gap return). Realistic fill model.
* **Accurate TWR:** Annualised using actual trading days, not configured period years.
* **Capital Gains Tax Simulation:** `apply_tax=True` deducts US tax on realised gains — 15% long-term (>365d), 25% short-term. After-tax TWR tracked separately.
* **Per-Trade Log:** Every closed trade shows entry/exit dates, hold days, gross return %, and tax paid.
* **DCA Support:** Periodic capital injections with separate TWR and Total ROI metrics.
* **Leverage & Decay Simulation:** Historical borrow rates by era, volatility drag modeled daily.

## Architecture
1. `BaseStrategy` — child classes implement `_add_indicator_logic(df)` to set `df['in_market']`.
2. `Backtester` — daily accounting, tax, equity curve, TWR/MaxDD for a single period.
3. `RollingBacktester` — parallel multi-period orchestration. Configurable `metric_key` / `metric_label`.
4. `cache_clear()` — flushes `DATA_CACHE` and `SIGNAL_CACHE`; call when changing indicator params or forcing a fresh download.

## Tax Rate Constants
```python
TAX_LONG_TERM_RATE  = 0.15   # 15% on gains held > 365 days
TAX_SHORT_TERM_RATE = 0.25   # 25% on gains held <= 365 days
```

---
*Ensure pandas, numpy, yfinance are installed before running.*


In [ ]:
from strat_backtest import *

# Optional: force fresh download + indicator recompute
# cache_clear()

## Single Backtest
Run individual strategies for a specific date range. `apply_tax=True` models after-tax returns.
The per-trade log is printed automatically and returned in `results['trade_log']`.


In [ ]:
# 1. Create environment
env = Backtester(
    base_ticker="^NDX",
    start_date="2000-01-01",
    period_years=25,
    leverage=3,
    initial_fund=10000,
    annual_dca=0,
    apply_tax=True,   # deduct capital gains tax on each realised gain
    verbose=True
)

# 2. Define strategies
strat_hold    = BuyAndHold()
strat_sma_atr = SMATrendFollowing(sma_window=200, atr_multiplier=2.5)

# 3. Run!
results_hold = env.run(strat_hold)
results_sma  = env.run(strat_sma_atr)

# Trade log available programmatically:
# results_sma['trade_log']  ->  list of dicts with entry/exit dates, hold_days, gross_ret_pct, tax_paid

## Single Backtest — VIX Filter Example


In [ ]:
env = Backtester(
    base_ticker="^NDX",
    start_date="2000-01-01",
    period_years=25,
    leverage=3,
    initial_fund=10000,
    annual_dca=0,
    apply_tax=False,
    verbose=True
)
strat_VIX = VolatilityFilter(name="VIX < 25", vix_threshold=25)
results_vix = env.run(strat_VIX)

## Rolling Backtest Suite (NDX — DCA)
Runs all strategies across hundreds of rolling start dates **in parallel**.
Returns a dict of DataFrames (one per leverage config) for further analysis.


In [ ]:
# 1. Define Dates
period_years = 26
end_date = pd.Timestamp.today() - pd.DateOffset(years=period_years)
monthly_start_dates = pd.date_range(
    start="1980-01-15",
    end=end_date,
    freq=pd.DateOffset(months=1)
)

# 2. Define Leverage Configs
my_configs = [
    {"name": "3x Leverage", "leverage": 3, "expense": 0.0095},
    {"name": "2x Leverage", "leverage": 2, "expense": 0.0095},
    {"name": "1x Leverage", "leverage": 1, "expense": 0.0020}
]

# 3. Define Strategies
strategies = [
    BuyAndHold(),
    SMATrendFollowing(sma_window=200, atr_multiplier=2.5),
    VolatilityFilter(name="VIX < 25", vix_threshold=25),
    EMACrossover(name="EMA 50/200"),
    RSIMeanReversion(name="RSI 30/70")
]

# 4. Run! (parallel across start dates via ThreadPoolExecutor)
results_dict = run_experiment_suite(
    configs=my_configs,
    strategies=strategies,
    start_dates=monthly_start_dates,
    period_years=26,
    initial_fund=10000,
    annual_dca=10000,
    apply_tax=False    # set True to see after-tax TWR across all strategies
)

## Rolling Backtest Suite (S&P 500 — Lump Sum)
Same suite run against `^GSPC` with no DCA.


In [ ]:
# 1. Define Dates
period_years = 26
end_date = pd.Timestamp.today() - pd.DateOffset(years=period_years)
monthly_start_dates = pd.date_range(
    start="1980-01-15",
    end=end_date,
    freq=pd.DateOffset(months=1)
)

# 2. Define Leverage Configs
my_configs = [
    {"name": "3x Leverage", "leverage": 3, "expense": 0.0095},
    {"name": "2x Leverage", "leverage": 2, "expense": 0.0095},
    {"name": "1x Leverage", "leverage": 1, "expense": 0.0020}
]

# 3. Define Strategies
strategies = [
    BuyAndHold(),
    SMATrendFollowing(sma_window=200, atr_multiplier=2.5),
    VolatilityFilter(name="VIX < 25", vix_threshold=25),
    EMACrossover(name="EMA 50/200"),
    RSIMeanReversion(name="RSI 30/70")
]

# 4. Run!
results_dict = run_experiment_suite(
    configs=my_configs,
    strategies=strategies,
    base_ticker="^GSPC",
    start_dates=monthly_start_dates,
    period_years=26,
    initial_fund=10000,
    annual_dca=0,
    apply_tax=False
)

## Advanced: Configurable Metric & Tax Comparison
Use `metric_key` / `metric_label` on `RollingBacktester` to rank by any result field,
or run side-by-side pre-tax vs after-tax comparisons.


In [ ]:
dates  = pd.date_range("2000-01-01", periods=24, freq=pd.DateOffset(months=6))
strats = [SMATrendFollowing(), BuyAndHold()]

# Pre-tax TWR
rb_pretax = RollingBacktester(
    start_dates=dates, period_years=10, leverage=3,
    apply_tax=False, metric_key="strategy_twr", metric_label="PreTax TWR"
)
df_pretax = rb_pretax.run(strats)

# After-tax TWR
rb_aftertax = RollingBacktester(
    start_dates=dates, period_years=10, leverage=3,
    apply_tax=True, metric_key="strategy_twr", metric_label="AfterTax TWR"
)
df_aftertax = rb_aftertax.run(strats)

col_pre  = "SMA 200 - ATR Buffer (x2.5) PreTax TWR (%)"
col_post = "SMA 200 - ATR Buffer (x2.5) AfterTax TWR (%)"
print("Pre-tax  avg TWR: %.2f%%" % df_pretax[col_pre].mean())
print("After-tax avg TWR: %.2f%%" % df_aftertax[col_post].mean())

# Rank by final_value instead of TWR
rb_fv = RollingBacktester(
    start_dates=dates, period_years=10, leverage=3,
    metric_key="final_value", metric_label="Final Value"
)
df_fv = rb_fv.run(strats)
df_fv.head()

## Cross-Signal Experiment: Trade TQQQ Using S&P 500 Signal

A common thesis is that the S&P 500 trend is a better (less noisy) leading indicator for broad-market risk.
This experiment tests whether using `^GSPC` signals to time `^NDX` (TQQQ) exposure improves outcomes
versus using NDX's own signal.

**Three setups compared** (SMA 200, ATR x2.5 strategy):
| Setup | Signal From | Returns From |
| :--- | :--- | :--- |
| NDX own signal | ^NDX | ^NDX |
| **NDX + GSPC signal** | **^GSPC** | **^NDX** |
| GSPC own signal | ^GSPC | ^GSPC |


In [ ]:
from strat_backtest import *
import pandas as pd

strat        = SMATrendFollowing(sma_window=200, atr_multiplier=2.5)
period_years = 26

# Shared rolling date range (overlapping history of both tickers)
cache_clear()
df_ndx  = get_cached_data('^NDX')
df_gspc = get_cached_data('^GSPC')
warmup  = max(df_ndx.index[0], df_gspc.index[0]) + pd.DateOffset(days=210)
end_lim = pd.Timestamp.today() - pd.DateOffset(years=period_years)
dates   = pd.date_range(start=warmup, end=end_lim, freq=pd.DateOffset(months=1))
print(f'{len(dates)} rolling windows: {dates[0].date()} to {dates[-1].date()}')

configs = [
    {'name': '3x', 'leverage': 3, 'expense': 0.0095},
    {'name': '2x', 'leverage': 2, 'expense': 0.0095},
    {'name': '1x', 'leverage': 1, 'expense': 0.0020},
]
setups = [
    ('NDX own signal',    '^NDX',  '^NDX'),
    ('NDX + GSPC signal', '^NDX',  '^GSPC'),  # <-- the key experiment
    ('GSPC own signal',   '^GSPC', '^GSPC'),
]

rows = []
for cfg in configs:
    for label, base, signal in setups:
        rb = RollingBacktester(
            start_dates=dates,
            base_ticker=base,
            signal_ticker=signal,   # <-- new parameter
            period_years=period_years,
            leverage=cfg['leverage'],
            expense_ratio=cfg['expense'],
            initial_fund=10000,
        )
        df = rb.run([strat])
        col_twr = f'{strat.name} TWR (%)'
        col_dd  = f'{strat.name} Max DD (%)'
        col_tr  = f'{strat.name} Total Trades'
        rows.append({
            'Leverage':   cfg['name'],
            'Setup':      label,
            'Avg TWR':    round(df[col_twr].mean(), 2),
            'Med TWR':    round(df[col_twr].median(), 2),
            'Worst TWR':  round(df[col_twr].min(), 2),
            'Worst DD':   round(df[col_dd].min(), 2),
            'Avg Trades': round(df[col_tr].mean(), 1),
        })

import pandas as pd
pd.DataFrame(rows)

### Key Takeaways

| Leverage | NDX own | NDX + GSPC | Delta TWR | Delta DD | Delta Trades |
| :--- | ---: | ---: | ---: | ---: | ---: |
| **3x** | 23.57% / -81.38% | 23.59% / -83.79% | **+0.02%** | -2.41% | **-3.2** |
| **2x** | 20.71% / -62.51% | 20.37% / -64.66% | -0.34% | -2.15% | **-3.2** |
| **1x** | 13.95% / -35.77% | 13.64% / -35.77% | -0.31% | 0.00% | **-3.2** |

**Format: `Avg TWR / Worst DD`**

**Conclusions:**
- At **3x**: GSPC signal produces virtually identical average TWR (+0.02%) with **~3 fewer trades** per period. Slightly worse worst-case drawdown because GSPC exits slightly later when NDX crashes fast.
- At **2x / 1x**: NDX own signal has a small edge (~0.3%) in average TWR, still fewer trades with GSPC signal.
- **Trade-off**: Fewer trades = lower real-world friction (commissions, bid-ask spread, less monitoring). Also tends toward longer holds, which can improve after-tax efficiency (more long-term gains).
- **Practical use**: GSPC signal is a reasonable alternative if you prefer a smoother, less reactive trigger for your TQQQ position.
